In [ ]:
#Import the necessary libraries
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from scipy.stats import qmc
import matplotlib.pyplot as plt

In [ ]:
X = np.array([[0.17152521, 0.34391687, 0.2487372 ],
             [0.24211446, 0.64407427, 0.27243281],
             [0.53490572, 0.39850092, 0.17338873],
             [0.49258141, 0.61159319, 0.34017639],
             [0.13462167, 0.21991724, 0.45820622],
             [0.34552327, 0.94135983, 0.26936348],
             [0.15183663, 0.43999062, 0.99088187],
             [0.64550284, 0.39714294, 0.91977134],
             [0.74691195, 0.28419631, 0.22629985],
             [0.17047699, 0.6970324,  0.14916943],
             [0.22054934, 0.29782524, 0.34355534],
             [0.66601366, 0.67198515, 0.2462953 ],
             [0.04680895, 0.23136024, 0.77061759],
             [0.60009728, 0.72513573, 0.06608864],
             [0.96599485, 0.86111969, 0.56682913],
             [0.151836, 0.439990, 0.990881],
             [0.938858, 0.600654, 0.074174],
             [0.49258141, 0.61159319, 0.34017639],
             [0.646905, 0.407973, 0.918519],
             [0.328441, 0.237981, 0.955325],
             [0.489946, 0.859065, 0.513374],
              [0.488396, 0.622280, 0.351098],
              [0.535020, 0.406231, 0.165253],
              [0.959804, 0.861929, 0.564256],
              [0.211973, 0.288673, 0.344302],
              [0.493645, 0.858708, 0.508075],
              [0.595241, 0.725296, 0.069826]

])
y = np.array([-0.1121222, -0.08796286, -0.11141465, -0.03483531,
              -0.04800758, -0.11062091, -0.39892551, -0.11386851,
              -0.13146061, -0.09418956, -0.04694741, -0.10596504,
              -0.11804826, -0.03637783, -0.05675837, -0.393904968,
              -0.048787319, -0.049699456, -0.111301025, -0.27073902468816746,
              -0.042885215750335756, -0.03996446428663898, -0.10865336055983822,
              -0.07328700046470267, -0.060295591161217105, -0.033330865358791414,
              -0.05581563156993059

])

#shape of X & y
print(X.shape)
print(y.shape)

In [ ]:
# GP setup
kernel = ConstantKernel(1.0) * Matern(
    nu=2.5,
    length_scale=[1.0, 1.0, 1.0],
    length_scale_bounds=(1e-3, 1e4)
)

gpr = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=20,
        alpha=1e-6,
        normalize_y=True
    )

# Fit on your 3D data
gpr.fit(X, y)


# Check what the GP learned
print("GP Model Diagnostics:")
print(f"  Kernel: {gpr.kernel_}")
print(f"  Length scales: {gpr.kernel_.k2.length_scale}")
print(f"  Training score: {gpr.score(X, y):.3f}") # Reshape y for scoring

# Define bounds for 3D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x1 bounds
    (X[:, 1].min(), X[:, 1].max()),  # x2 bounds
    (X[:, 2].min(), X[:, 2].max())   # x3 bounds
]
print(f"  Bounds: {bounds}")

# Diagnostics
print(f"Length scales: {gpr.kernel_.k2.length_scale}")
print(f"Training R²: {gpr.score(X, y):.3f}")

# Generate 3D candidates
sampler = qmc.LatinHypercube(d=3)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
)

# Predict on candidates
y_pred, y_std = gpr.predict(X_candidates, return_std=True)

# UCB acquisition function
kappa = 2.0
ucb = y_pred + kappa * y_std

# Find best point
best_idx = np.argmax(ucb)
x_next = X_candidates[best_idx]

print(f"\nNext Point to Sample:")
print(f"  X = {x_next}")  # Should show 3 values
print(f"  Predicted y = {y_pred[best_idx]:.4f}")
print(f"  Uncertainty = {y_std[best_idx]:.4f}")
print(f"  UCB score = {ucb[best_idx]:.4f}")

# Show top 5 candidates
top5_idx = np.argsort(ucb)[-5:][::-1]
print(f"\nTop 5 Candidates:")
for i, idx in enumerate(top5_idx, 1):
    print(f"  {i}. X={X_candidates[idx]}, "
          f"pred={y_pred[idx]:.3f}, std={y_std[idx]:.3f}, ucb={ucb[idx]:.3f}")
    # add X candidate to X
    X = np.vstack((X, X_candidates[idx]))
    # Reshape y to (N, 1) if it's not already, and reshape the new prediction to (1, 1)
    # It's better to ensure y1 is always (N, 1) from the start, but for an immediate fix:
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    y = np.vstack((y, y_pred[idx].reshape(-1, 1)))

In [ ]:
#shape of X & y
print(X.shape)
print(y.shape)

Trying PyTorch & TensorFlow or this week

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import qmc

In [ ]:
# STEP 1: Define PyTorch Model
# ========================================
class NNSurrogate(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 32], dropout=0.2):
        super(NNSurrogate, self).__init__()

        layers = []
        prev_size = input_dim

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#========================================
# STEP 2: Train on Your Data
# ========================================
def train_on_your_data(X, y, epochs=1000, lr=0.001):
    """
    Train PyTorch model on your X and y
    """
    n_samples, input_dim = X.shape
    print(f"Training on {n_samples} samples, {input_dim}D")

    # Convert to tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).reshape(-1, 1)

    # Normalize
    X_mean, X_std = X_tensor.mean(0), X_tensor.std(0) + 1e-8
    y_mean, y_std = y_tensor.mean(), y_tensor.std() + 1e-8

    X_norm = (X_tensor - X_mean) / X_std
    y_norm = (y_tensor - y_mean) / y_std

    # Create model
    model = NNSurrogate(input_dim, hidden_sizes=[64, 32], dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)

    # Train
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_norm)
        loss = criterion(predictions, y_norm)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # Evaluate on training data
    model.eval()
    with torch.no_grad():
        train_pred = model(X_norm)
        train_pred_denorm = train_pred * y_std + y_mean
        mse = ((train_pred_denorm - y_tensor) ** 2).mean().item()
        r2 = 1 - mse / y_tensor.var().item()

    print(f"\nTraining Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R²: {r2:.4f}")

    return model, X_mean, X_std, y_mean, y_std

In [ ]:
# Train the model
model, X_mean, X_std, y_mean, y_std = train_on_your_data(X, y, epochs=1000)

In [ ]:
# ========================================
# STEP 3: Predict on New Points
# ========================================
def predict_new_points(model, X_new, X_mean, X_std, y_mean, y_std):
    """
    Predict on new X points
    """
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_new)
        X_norm = (X_tensor - X_mean) / X_std
        y_pred_norm = model(X_norm)
        y_pred = y_pred_norm * y_std + y_mean

    return y_pred.numpy().flatten()

# generating new candidates

n_dims = X.shape[1]

# Define bounds for 3D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x1 bounds
    (X[:, 1].min(), X[:, 1].max()),   # x2 bounds
    (X[:, 2].min(), X[:, 2].max())    # x3 bounds
]

# Generate candidates
sampler = qmc.LatinHypercube(d=n_dims)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
 )

y_pred = predict_new_points(model, X_candidates, X_mean, X_std, y_mean, y_std)
print(f"\nPredictions on new points:")
print(f"X_new: {X_candidates}")
print(f"y_pred: {y_pred}")

#print first X candidate and y pred value
print(f"First X candidate: {X_candidates[0]}")
print(f"First y prediction: {y_pred[0]}")